# ASAP8 VIP somatic voltage characterization

Single-session analysis of ASAP8+ VIP interneurons during passive Detection of Change.

The notebook is organized around the opening electrophysiology section of the lab meeting:

1. recorded neurons and visually inspectable dF/F traces;
2. isolated spike-like waveform features;
3. event rate, compound events, and sustained depolarization;
4. spike/event synchrony within and across DMDs.

All session selection, paths, cortical depths, and acquisition metadata are resolved through `VIPSessionRegistry`, the selected asset, and `asset.qc_dir`. Outputs retain `session_id`, `dmd`, and `roi` so longitudinal ROI identities can be added later without changing the analysis tables.

## 0. Imports and plotting style

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

from pathlib import Path
from datetime import datetime
import json
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from scipy import ndimage, signal
from scipy.signal import find_peaks
from IPython.display import display, HTML

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))

NAVY = "#003057"
BLUE = "#2A7DE2"
PEACH = "#EBA287"
GRAY = "#737373"
LIGHT_GRAY = "#D9D9D9"
DMD_COLORS = {1: BLUE, 2: PEACH}

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

from vip_slap2_analysis.utils.utils import save_figure
savepath = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots"

## 1. Select one session through the registry

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_MOUSE = 852835
SESSION_IDX = -3  # None selects the latest registry session

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"

SAVE_FIGURES = True
SAVE_TABLES = True

registry = VIPSessionRegistry.from_basepath(BASE_PATH)
session_df = registry.sessions(
    subject_ids=[TARGET_MOUSE],
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
).sort_values("session_date").reset_index(drop=True)

assets = [registry.resolve_assets(row) for _, row in session_df.iterrows()]
asset = assets[SESSION_IDX]

show_cols = [c for c in [
    "session_id", "subject_id", "session_date", "session_type",
    "dmd1_depth", "dmd2_depth", "quality",
] if c in session_df.columns]

display(session_df[show_cols])
print("Selected:", asset.session_id)

## 2. Resolve paths and acquisition metadata from the asset

In [ ]:
VOLTAGE_QC_PATH = (
    asset.qc_dir
    / "voltage"
    / f"voltage_extraction_qc_{TRACE_VARIANT}.json"
)

with open(VOLTAGE_QC_PATH, "r") as f:
    voltage_qc = json.load(f)

FS = float(voltage_qc["sample_rate_hz"])
SUMMARY_PATH = Path(voltage_qc["summary_mat"])
TRACE_H5_PATH = (
    asset.derived_dir
    / "voltage"
    / f"voltage_session_traces_{TRACE_VARIANT}.h5"
)

OUT_DIR = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV")
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
for directory in (OUT_DIR, FIG_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

depths = {
    1: float(asset.metadata["dmd1_depth"]),
    2: float(asset.metadata["dmd2_depth"]),
}

print(f"Session:     {asset.session_id}")
print(f"Sample rate: {FS:,.3f} Hz")
print(f"Trace:       {TRACE_H5_PATH}")
print(f"Summary:     {SUMMARY_PATH}")
print(f"Depths:      DMD1={depths[1]:.0f} µm, DMD2={depths[2]:.0f} µm below pia")
print(f"Output:      {OUT_DIR}")

## 3. Trace and summary readers

In [ ]:
def _as_scalar(x):
    arr = np.asarray(x).squeeze()
    if arr.size == 1:
        value = arr.reshape(-1)[0]
        return value.item() if isinstance(value, np.generic) else value
    return arr


def _orient_trace_dataset(ds, expected_n_rois):
    arr = np.asarray(ds[()], dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D trace dataset, got {arr.shape}")
    if arr.shape[0] == expected_n_rois:
        return arr
    if arr.shape[1] == expected_n_rois:
        return arr.T
    raise ValueError(
        f"Neither axis matches expected_n_rois={expected_n_rois}; shape={arr.shape}"
    )


def _read_trial_timing(summary_path, n_trials, fs, trial_lengths):
    starts = None
    ends = None
    with h5py.File(summary_path, "r") as f:
        if "summary/trialTable/trialStartTimeInferred" in f:
            starts = np.asarray(
                f["summary/trialTable/trialStartTimeInferred"][()], dtype=float
            ).reshape(-1)[:n_trials]
        if "summary/trialTable/trialEndTimeFromPC" in f:
            ends = np.asarray(
                f["summary/trialTable/trialEndTimeFromPC"][()], dtype=float
            ).reshape(-1)[:n_trials]

    total = int(np.sum(trial_lengths))
    compressed_time_sec = np.arange(total, dtype=np.float64) / float(fs)
    acquisition_time_sec = np.empty(total, dtype=np.float64)
    trial_id = np.empty(total, dtype=np.int32)
    trial_slices = []

    cursor = 0
    for i, n in enumerate(trial_lengths):
        n = int(n)
        sl = slice(cursor, cursor + n)
        trial_id[sl] = i + 1
        trial_slices.append({"trial": i + 1, "start": cursor, "stop": cursor + n})
        if starts is not None and np.all(np.isfinite(starts)):
            start_sec = (starts[i] - starts[0]) * 86400.0
            acquisition_time_sec[sl] = start_sec + np.arange(n) / float(fs)
        else:
            acquisition_time_sec[sl] = compressed_time_sec[sl]
        cursor += n

    return {
        "compressed_time_sec": compressed_time_sec,
        "acquisition_time_sec": acquisition_time_sec,
        "trial_id": trial_id,
        "trial_slices": trial_slices,
        "trial_start_matlab_datenum": starts,
        "trial_end_matlab_datenum": ends,
    }

In [ ]:
def load_derived_voltage_traces(
    summary_path,
    trace_h5_path,
    fs,
    signal="dff",
):
    """Load processed voltage-session H5 data as DMD -> ROI x time."""

    with h5py.File(summary_path, "r") as sf:
        n_rois = np.asarray(
            sf["summary/nAnalysisROIs"][()],
            dtype=int,
        ).reshape(-1)

        trial_lengths = np.asarray(
            sf["summary/trialLineRanges/trialGlobalNLines"][()],
            dtype=int,
        ).reshape(-1)

    traces = {}

    with h5py.File(trace_h5_path, "r") as tf:
        for dmd, expected_n_rois in enumerate(n_rois, start=1):
            dataset_path = f"DMD{dmd}/{signal}"

            if dataset_path not in tf:
                raise KeyError(
                    f"{dataset_path!r} not found in {trace_h5_path}. "
                    f"Root groups: {list(tf.keys())}"
                )

            traces[dmd] = _orient_trace_dataset(
                tf[dataset_path],
                int(expected_n_rois),
            )

    sample_counts = {x.shape[1] for x in traces.values()}
    if len(sample_counts) != 1:
        raise ValueError(
            f"DMD datasets have different sample counts: {sample_counts}"
        )

    n_samples = sample_counts.pop()

    summary_n_samples = int(trial_lengths.sum())

    if summary_n_samples != n_samples:
        delta = summary_n_samples - n_samples

        print(
            "WARNING: extraction-summary and processed-H5 sample counts differ.\n"
            f"  Summary:       {summary_n_samples:,} samples\n"
            f"  Processed H5:  {n_samples:,} samples\n"
            f"  Difference:    {delta:,} samples "
            f"({delta / fs:.3f} s at {fs:.3f} Hz)\n"
            "\n"
            "Ignoring summary-derived trial boundaries and treating the "
            "processed H5 as one continuous recording."
        )

        timing = {
            "trial_slices": None,
            "trial_start_sec": None,
            "trial_stop_sec": None,
        }

    else:
        timing = _read_trial_timing(
            summary_path,
            len(trial_lengths),
            fs,
            trial_lengths,
        )

    return traces, {
        "mode": "processed_trial_concatenated",
        "signal": signal,
        "trial_lengths_samples": trial_lengths,
        **timing,
    }

## 4. Reference images, ROI masks, and ROI manifest

In [ ]:
def orient_slap2_image_for_display(image):
    return np.flipud(np.asarray(image).T)


def orient_slap2_masks_for_display(masks):
    return np.asarray(masks).transpose(0, 2, 1)[:, ::-1, :]


def read_ref_image(summary_path, dmd):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/refIM"][dmd - 1, 0]
        image = np.asarray(f[ref][()], dtype=np.float32)
    return orient_slap2_image_for_display(image)


def read_roi_masks(summary_path, dmd):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/masks"][dmd - 1, 0]
        masks = np.asarray(f[ref][()], dtype=bool)
    return orient_slap2_masks_for_display(masks)


traces, trace_info = load_derived_voltage_traces(
    SUMMARY_PATH,
    TRACE_H5_PATH,
    FS,
    signal="dff",
)

segment_slices = (
    [slice(x["start"], x["stop"]) for x in trace_info["trial_slices"]]
    if trace_info["trial_slices"] is not None
    else [slice(0, traces[1].shape[1])]
)

duration_s = traces[1].shape[1] / FS
print(f"Trace mode: {trace_info['mode']}")
print(f"Duration:   {duration_s / 60:.2f} min")
for dmd, x in traces.items():
    print(f"DMD{dmd}: {x.shape[0]} ROIs x {x.shape[1]:,} samples")

reference_images = {dmd: read_ref_image(SUMMARY_PATH, dmd) for dmd in traces}
roi_masks = {dmd: read_roi_masks(SUMMARY_PATH, dmd) for dmd in traces}

manifest_rows = []
for dmd, masks in roi_masks.items():
    for roi, mask in enumerate(masks):
        yy, xx = np.nonzero(mask)
        manifest_rows.append({
            "subject_id": asset.subject_id,
            "session_id": asset.session_id,
            "dmd": dmd,
            "roi": roi,
            "depth_um": depths[dmd],
            "roi_area_px": int(mask.sum()),
            "centroid_x_px": float(np.mean(xx)),
            "centroid_y_px": float(np.mean(yy)),
        })

roi_manifest = pd.DataFrame(manifest_rows)
display(roi_manifest)

In [ ]:
fig, axes = plt.subplots(1, len(traces), figsize=(5.2 * len(traces), 4.6))
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    image = reference_images[dmd]
    masks = roi_masks[dmd]
    lo, hi = np.nanpercentile(image, [1, 99.7])
    if image.ndim>2:
        im = image[:,:,0]
        ax.imshow(im, cmap="gray", vmin=lo, vmax=hi)
    else:
        ax.imshow(image, cmap="gray", vmin=lo, vmax=hi) 
    for roi, mask in enumerate(masks):
        ax.contour(mask, levels=[0.5], colors=[DMD_COLORS[dmd]], linewidths=1.0)
        yy, xx = np.nonzero(mask)
        ax.text(
            np.mean(xx), np.mean(yy), str(roi),
            color="white", fontsize=9, ha="center", va="center",
            bbox={"facecolor": NAVY, "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
        )
    ax.set_title(f"DMD{dmd} · {depths[dmd]:.0f} µm below pia")
    ax.axis("off")

fig.suptitle("ASAP8 somatic ROIs", color=NAVY,fontsize=18,y=0.95)
fig.tight_layout()


## 5. Visual inspection of the dF/F traces

The first plot compresses the complete recording for drift and activity-regime inspection. The second uses one editable time window for direct waveform comparison.

In [ ]:
OVERVIEW_BIN_MS = 2.0
ZOOM_SEC = (10.0, 15.0)

In [ ]:
fig, axes = plt.subplots(
    len(traces), 1,
    figsize=(13, 2.5 + 1.0 * sum(x.shape[0] for x in traces.values())),
    sharex=True,
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    x = traces[dmd]
    step = max(1, int(round(OVERVIEW_BIN_MS / 1000 * FS)))
    n = x.shape[1] // step
    reduced = x[:, :n * step].reshape(x.shape[0], n, step).mean(axis=2)
    t = np.arange(n) * step / FS / 60

    robust = np.nanpercentile(reduced, 95, axis=1) - np.nanpercentile(reduced, 5, axis=1)
    offsets = np.r_[0, np.cumsum(np.maximum(robust[:-1], np.nanmedian(robust)) * 3)]
    for roi, row in enumerate(reduced):
        centered = row - np.nanmedian(row)
        ax.plot(t, centered + offsets[roi], lw=0.55, color=DMD_COLORS[dmd])
        ax.text(t[0] - 0.01 * t[-1], offsets[roi], f"ROI {roi}", ha="right", va="center")

    ax.set_yticks([])
    ax.set_ylabel(f"DMD{dmd}\n{depths[dmd]:.0f} µm below pia")
    ax.set_title(f"DMD{dmd}: full-session dF/F")

axes[-1].set_xlabel("Session time (min)")
fig.tight_layout()
plt.show()

In [ ]:
t0, t1 = ZOOM_SEC
i0, i1 = int(t0 * FS), int(t1 * FS)

fig, axes = plt.subplots(
    len(traces), 1,
    figsize=(13, 2.3 + 0.85 * sum(x.shape[0] for x in traces.values())),
    sharex=True,
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    window = traces[dmd][:, i0:i1]
    t = np.arange(i0, i1) / FS
    scale = np.nanmedian(
        np.nanpercentile(window, 95, axis=1) - np.nanpercentile(window, 5, axis=1)
    )
    offsets = np.arange(window.shape[0]) * scale * 1.75
    for roi, row in enumerate(window):
        ax.plot(t, row - np.nanmedian(row) + offsets[roi], lw=0.65, color=DMD_COLORS[dmd])
        ax.text(t0 - 0.01 * (t1 - t0), offsets[roi], f"ROI {roi}", ha="right", va="center")
        sample_rate = 10_700
        peaks = find_peaks(row,
                   distance = sample_rate*0.001,
                   width = sample_rate*0.002,
                   height = np.mean(row)+1.5*np.std(row,ddof=1),
#                    prominence = 0.2
                  )
        indx = peaks[0]
        amps = peaks[1]['peak_heights']
        ax.scatter(t[indx],amps+offsets[roi],marker='v',color='k',s=10,label='Detected spike',zorder=10)
        
    ax.set_yticks([])
    ax.set_ylabel(f"DMD{dmd}\n{depths[dmd]:.0f} µm")
    ax.set_title(f"DMD{dmd}: {t0:g}–{t1:g} s")
    

axes[-1].set_xlabel("Time (s)")
fig.tight_layout()


In [ ]:
t0, t1 = ZOOM_SEC
i0, i1 = int(t0 * FS), int(t1 * FS)

fig, axes = plt.subplots(
    2, 1,
    figsize=(9, 4),
    sharex=True,
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    
    sns.despine(ax = ax,bottom=True)
    ax.set_xticks([])
    ax.set_xticklabels([])
    
    for spine in ['left']:
        ax.spines[spine].set_linewidth(2)
    
    window = traces[dmd][0:1, i0:i1]
    t = np.arange(i0, i1) / FS
    scale = np.nanmedian(
        np.nanpercentile(window, 95, axis=1) - np.nanpercentile(window, 5, axis=1)
    )
    offsets = np.arange(window.shape[0]) * scale * 1.75
    for roi, row in enumerate(window):
        ax.plot(t, row - np.nanmedian(row) + offsets[roi], lw=0.75, color=DMD_COLORS[dmd])
        sample_rate = 10_700
        peaks = find_peaks(row,
                   distance = sample_rate*0.004,
                   width = sample_rate*0.001,
                   height = np.mean(row)+2*np.std(row,ddof=1),
                   prominence = 0.2
                  )
        indx = peaks[0]
        amps = peaks[1]['peak_heights']
#         ax.scatter(t[indx],amps+offsets[roi],marker='v',color='k',s=10,label='Detected spike',zorder=10)
    
    ax.set_yticks([])
    ax.set_ylabel(f"DMD{dmd} ROI {roi}\n{depths[dmd]:.0f} µm below pia")
axes[0].set_title('Example ASAP8+ VIP interneuron activity',fontsize=18)
fig.tight_layout()
plt.show()

In [ ]:
# Scale bar on lower-right axis
ax = axes[-1]

TIME_BAR_S = 1.0
DFF_BAR = 0.2

x_range = t1 - t0
ymin, ymax = ax.get_ylim()
y_range = ymax - ymin

x_right = t1 - 0.035 * x_range
x_left = x_right - TIME_BAR_S
y_bottom = ymin + 0.09 * y_range
y_top = y_bottom + DFF_BAR

bar_color = "#183B56"

ax.plot(
    [x_left, x_right],
    [y_bottom, y_bottom],
    color=bar_color,
    lw=2.2,
    solid_capstyle="butt",
    zorder=20,
    clip_on=False,
)
ax.plot(
    [x_right, x_right],
    [y_bottom, y_top],
    color=bar_color,
    lw=2.2,
    solid_capstyle="butt",
    zorder=20,
    clip_on=False,
)

ax.text(
    (x_left + x_right) / 2,
    y_bottom - 0.035 * y_range,
    f"{TIME_BAR_S:g} s",
    ha="center",
    va="top",
    fontsize=9,
    color=bar_color,
)
ax.text(
    x_right + 0.012 * x_range,
    (y_bottom + y_top) / 2,
    f"{DFF_BAR:g} ΔF/F",
    ha="left",
    va="center",
    fontsize=9,
    color=bar_color,
)

### To Do:
- Bandpass filter each trace
- Run find_peaks
- Develop burst detection method - maybe not even worth trying to parse single spikes in these events
- Develop single spike metrics


In [ ]:
from scipy.signal import butter, sosfiltfilt
from scipy.signal import find_peaks

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    # Design using 'sos' for stability
    sos = butter(order, [lowcut, highcut], btype='band', fs=fs, output='sos')
    # Use sosfiltfilt for zero phase distortion
    return sosfiltfilt(sos, data)

In [ ]:
sample_rate = 10_700
window = (sample_rate*10,sample_rate*15)
data = traces[2][3,window[0]:window[1]]

filtered = bandpass_filter(data, 20, 2400, sample_rate, order=3)

In [ ]:
peaks = find_peaks(filtered,
                   distance = sample_rate*0.001,
                   width = sample_rate*0.001,
                   height = np.mean(filtered)+2.0*np.std(filtered,ddof=1),
#                    prominence = 0.05
                  )
indx = peaks[0]
amps = peaks[1]['peak_heights']

In [ ]:
fig,ax=plt.subplots()

t = np.linspace(window[0]/sample_rate,window[1]/sample_rate,len(data))

ax.plot(t,data,lw=0.5,label='DMD2 ROI 3 somatic \u0394F/F$_{0}$')
ax.plot(t,filtered,lw=0.5,label='Band-pass filtered \u0394F/F$_{0}$ (20-1000Hz)')
ax.scatter(t[indx],data[indx],marker='v',color='k',s=5,label='Detected spike')

ax.set_xlim(10,12.5)

ax.set_xlabel('Time (s)')
ax.set_ylabel('\u0394F/F$_{0}$')
ax.legend(fontsize=8,frameon=False)
fig.tight_layout()

In [ ]:
sample_rate = 10_700
t1,t2 = (54,56)
dmd = 2
roi = 3 
colors = ['#eaa186','#4379bc']
window = (sample_rate*t1,sample_rate*t2)
data = pd.DataFrame(traces[dmd][roi,window[0]:window[1]]).rolling(20,min_periods=1).mean()

fig,ax=plt.subplots(figsize=(2,1))

t = np.linspace(t1,t2,len(data))

ax.plot(t,data,lw=0.75,color=colors[dmd-1])

ax.set_xticks([])
ax.set_yticks([])
sns.despine(left=True,bottom=True)

# ax.set_ylim(-0.5,1.1)

fig.tight_layout()
fig.subplots_adjust(left=0, bottom=0, right=1, top=1)
filen = f'DMD{dmd}_ROI{roi}_TraceSegment'
# save_figure(fig,os.path.join(savepath,filen),formats=['.pdf'],dpi=300)

In [ ]:
print(f'Data min = {np.min(data)}')
print(f'Data max = {np.max(data)}')
print(f'Data range = {np.max(data)-np.min(data)}')

### Single-spike plot

In [ ]:
dmd = 1
roi = 0

trace = traces[dmd][roi,:]
sample_rate = 10_700

window = (sample_rate*10,sample_rate*11)

roi_data = trace[window[0]:window[1]]

In [ ]:
fig,ax = plt.subplots(figsize=(3.5,3.5))

sns.despine(left=True,bottom=True)

t = np.linspace(window[0]/sample_rate,window[1]/sample_rate,len(roi_data))

ax.plot(t,roi_data,lw=3,color='fuchsia')
ax.set_xlim(10.15,10.45)

ax.set_xticks([])
ax.set_yticks([])

fig.tight_layout()

filen = 'Intro_panel1'
# save_figure(fig,os.path.join(FIG_DIR,filen),formats=['.pdf'],dpi=300)

In [ ]:
dmds = [1,2]
colors = ['#eaa186','#4379bc']
for i,dmd in enumerate(dmds):
    roi_traces = traces[dmd]
    
    for roi in roi_traces:
        fig,ax=plt.subplots(figsize=(2,1))
        
        sample_rate = 10_700
        window = (sample_rate*10,sample_rate*11)
        roi_data = roi[window[0]:window[1]]
        t = np.linspace(0,(window[1]/sample_rate)-(window[0]/sample_rate),len(roi_data))
        
        ax.plot(t,roi_data,color=colors[i])

### Spike detection/dynamics plot

In [ ]:
sample_rate = 10_700
window = (sample_rate*130,sample_rate*135)

fig,axes=plt.subplots(2,figsize=(5,4))

dmds = [1,2]
rois = [1,0]

for i,ax in enumerate(axes.flatten()):
    
    data = traces[dmds[i]][rois[i],window[0]:window[1]]
    t = np.linspace((window[0]-window[0])/sample_rate,(window[1]-window[0])/sample_rate,len(data))
    
    peaks = find_peaks(data,
                   distance = sample_rate*0.001,
                   width = sample_rate*0.001,
                   height = np.mean(data)+1.75*np.std(data,ddof=1),
                   prominence = 0.1
                  )
    indx = peaks[0]
    amps = peaks[1]['peak_heights']
    
    plot_data = pd.DataFrame(data).rolling(1,min_periods=1).mean()
    
    ax.plot(t,plot_data,lw=0.75,color=colors[i])
    ax.scatter(t[indx],amps+0.01,marker='v',color='fuchsia',s=7,label='Detected spike',zorder=10)

    ax.set_xlim(1,4)
    ax.set_ylim(-0.25,0.95)
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)
    
    sns.despine(ax=ax,left=True,bottom=True)
    
axes[0].plot([3.3,3.55],[0.3,0.3],color='k',lw=2,solid_capstyle='round')
axes[0].text(3.3,0.21,'250 ms',fontsize=8)
axes[0].plot([3.3,3.3],[0.3,0.5],color='k',lw=2,solid_capstyle='round')
axes[0].text(3.075,0.33,'0.5\n\u0394F/F$_{0}$',fontsize=8,rotation=0)
axes[0].legend(fontsize=8,handletextpad=0.01,markerscale=2)

fig.tight_layout()
fig.subplots_adjust(hspace=0.0,left=0, bottom=0, right=1, top=1)

filen = 'spike_detection'
save_figure(fig,os.path.join(FIG_DIR,filen),formats=['.pdf'],dpi=300)